In [21]:
import numpy as np
import pandas as pd
import plotly.express as px
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

In [22]:
# ==========================================
# 1. FUNÇÃO DE TRATAMENTO DE DADOS SEGURO
# ==========================================
def limpar_valor_brasileiro(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    val_str = str(val).strip()
    # Se houver vírgula, trata a formatação brasileira (ex: "527.044,36" -> "527044.36")
    if ',' in val_str:
        val_str = val_str.replace('.', '').replace(',', '.')
    return float(val_str)

meses_map = {
    'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04',
    'mai': '05', 'jun': '06', 'jul': '07', 'ago': '08',
    'set': '09', 'out': '10', 'nov': '11', 'dez': '12'
}

def parse_portuguese_date(date_str):
    mes, ano = str(date_str).strip().split('/')
    mes_num = meses_map[mes.lower()]
    return f"20{ano}-{mes_num}-01"

# ==========================================
# 2. CARREGAMENTO E TRANSFORMAÇÃO DA TABELA
# ==========================================
df_raw = pd.read_excel('./DF/total_mensal_fixed.xlsx')

# Transposição correta para formato vertical
df_long = df_raw.T.reset_index()
df_long.columns = ['data_str', 'valor']

# Aplicação da conversão limpa
df_long['ds'] = pd.to_datetime(df_long['data_str'].apply(parse_portuguese_date))
df_long['valor'] = df_long['valor'].apply(limpar_valor_brasileiro)

# Organização do índice temporal contínuo (2013-2020)
df_series = df_long.dropna(subset=['valor']).drop_duplicates(subset=['ds']).set_index('ds')['valor'].asfreq('MS')

# ==========================================
# 3. PASSO DE AVALIAÇÃO/TESTE (TREINO: 2013-2018 | TESTE: 2019-2020)
# ==========================================
train_val = df_series['2013-01-01':'2018-12-01']
test_val = df_series['2019-01-01':'2020-12-01']
steps_val = len(test_val)

# A. Teste ARIMA
fit_arima_val = ARIMA(train_val, order=(1, 1, 1)).fit()
pred_arima_val = fit_arima_val.forecast(steps=steps_val)

# B. Teste SARIMA
fit_sarima_val = SARIMAX(train_val, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12),
                         enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
pred_sarima_val = fit_sarima_val.forecast(steps=steps_val)

# C. Teste Random Forest
def create_rf_features(data):
    df_feat = pd.DataFrame({'target': data})
    df_feat['month'] = df_feat.index.month
    df_feat['lag_1'] = df_feat['target'].shift(1)
    df_feat['lag_2'] = df_feat['target'].shift(2)
    df_feat['lag_12'] = df_feat['target'].shift(12)
    return df_feat.dropna()

df_rf_val = create_rf_features(train_val)
rf_val = RandomForestRegressor(n_estimators=200, random_state=42).fit(df_rf_val.drop('target', axis=1), df_rf_val['target'])

pred_rf_val = []
hist_val = train_val.tolist()
for date in test_val.index:
    feat = pd.DataFrame([[date.month, hist_val[-1], hist_val[-2], hist_val[-12]]],
                        columns=['month', 'lag_1', 'lag_2', 'lag_12'])
    p = rf_val.predict(feat)[0]
    pred_rf_val.append(p)
    hist_val.append(p)

# Cálculo e exibição das métricas
def calc_metricas(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {'Modelo': name, 'MAE (Tons)': round(mae, 2), 'RMSE (Tons)': round(rmse, 2), 'MAPE (%)': round(mape, 2)}

df_metricas = pd.DataFrame([
    calc_metricas(test_val, pred_rf_val, 'Random Forest'),
    calc_metricas(test_val, pred_arima_val, 'ARIMA'),
    calc_metricas(test_val, pred_sarima_val, 'SARIMA')
])

print("\n--- AVALIAÇÃO DE PRECISÃO NO PERÍODO DE TESTE (2019-2020) ---")
print(df_metricas.to_string(index=False))

# ==========================================
# 4. TREINAMENTO COMPLETO (2013-2020) E PREVISÃO (2021-2027)
# ==========================================
train_full = df_series['2013-01-01':'2020-12-01']
future_dates = pd.date_range(start='2021-01-01', end='2027-12-01', freq='MS')
steps_full = len(future_dates)

# ARIMA Final
arima_full = ARIMA(train_full, order=(1, 1, 1)).fit()
pred_arima_full = arima_full.forecast(steps=steps_full)

# SARIMA Final
sarima_full = SARIMAX(train_full, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12),
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
pred_sarima_full = sarima_full.forecast(steps=steps_full)

# Random Forest Final
df_rf_full = create_rf_features(train_full)
rf_full = RandomForestRegressor(n_estimators=200, random_state=42).fit(df_rf_full.drop('target', axis=1), df_rf_full['target'])

pred_rf_full = []
hist_full = train_full.tolist()
for date in future_dates:
    feat = pd.DataFrame([[date.month, hist_full[-1], hist_full[-2], hist_full[-12]]],
                        columns=['month', 'lag_1', 'lag_2', 'lag_12'])
    p = rf_full.predict(feat)[0]
    pred_rf_full.append(p)
    hist_full.append(p)

# Tabela de saída consolidada
df_resultado = pd.DataFrame({
    'ds': future_dates.strftime('%Y-%m-%d %H:%M:%S'),
    'Previsao_RandomForest': pred_rf_full,
    'Previsao_ARIMA': pred_arima_full.values,
    'Previsao_SARIMA': pred_sarima_full.values
})

df_resultado.to_csv("./DF/previsoes_RSU_SP_2021_2027.csv", index=False)

# ==========================================
# 5. GRÁFICO INTERATIVO PLOTLY
# ==========================================
ancora_dt = pd.to_datetime('2020-12-01')
ancora_val = train_full.iloc[-1]

df_real_p = train_full.reset_index()
df_real_p.columns = ['ds', 'valor']
df_real_p['Serie'] = 'Histórico Real (2013-2020)'

df_rf_p = pd.concat([pd.DataFrame({'ds': [ancora_dt], 'valor': [ancora_val]}), pd.DataFrame({'ds': future_dates, 'valor': pred_rf_full})])
df_rf_p['Serie'] = 'Previsão Random Forest'

df_arima_p = pd.concat([pd.DataFrame({'ds': [ancora_dt], 'valor': [ancora_val]}), pd.DataFrame({'ds': future_dates, 'valor': pred_arima_full.values})])
df_arima_p['Serie'] = 'Previsão ARIMA'

df_sarima_p = pd.concat([pd.DataFrame({'ds': [ancora_dt], 'valor': [ancora_val]}), pd.DataFrame({'ds': future_dates, 'valor': pred_sarima_full.values})])
df_sarima_p['Serie'] = 'Previsão SARIMA'

df_plot_full = pd.concat([df_real_p, df_rf_p, df_arima_p, df_sarima_p], ignore_index=True)

fig = px.line(
    df_plot_full,
    x='ds',
    y='valor',
    color='Serie',
    title='<b>Coleta Total de RSU em SP: Histórico Confiável (2013-2020) e Projeções (2021-2027)</b>',
    labels={'ds': 'Ano/Mês', 'valor': 'Toneladas Coletadas', 'Serie': 'Legenda'},
    color_discrete_map={
        'Histórico Real (2013-2020)': '#111111',
        'Previsão Random Forest': '#2ca02c',
        'Previsão ARIMA': '#ff7f0e',
        'Previsão SARIMA': '#1f77b4'
    }
)

fig.update_traces(line=dict(width=2.5))
fig.add_vline(x=pd.to_datetime('2021-01-01').timestamp() * 1000, line_dash="dash", line_color="red",
              annotation_text="Início da Inconsistência nos Dados", annotation_position="top left")

fig.update_layout(
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True, tickformat=",.0f"),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


--- AVALIAÇÃO DE PRECISÃO NO PERÍODO DE TESTE (2019-2020) ---
       Modelo  MAE (Tons)  RMSE (Tons)  MAPE (%)
Random Forest    18561.28     22314.90      3.90
        ARIMA    20246.60     25512.99      4.41
       SARIMA    21016.24     25881.78      4.51


In [23]:
# ==========================================
# GRÁFICO DE VALIDAÇÃO: REAL VS PREVISTO (2019-2020)
# ==========================================

# 1. Estruturação do DataFrame de comparação para o teste
df_comparacao_val = pd.DataFrame({
    'ds': test_val.index,
    'Real (2019-2020)': test_val.values,
    'Random Forest': pred_rf_val,
    'ARIMA': pred_arima_val.values,
    'SARIMA': pred_sarima_val.values
})

# 2. Conversão para o formato longo (melt) ideal do Plotly Express
df_val_long = df_comparacao_val.melt(
    id_vars=['ds'], 
    var_name='Modelo', 
    value_name='Toneladas'
)

# 3. Construção do gráfico de validação
fig_val = px.line(
    df_val_long,
    x='ds',
    y='Toneladas',
    color='Modelo',
    title='<b>Validação de Precisão: Dados Reais vs. Modelos (2019-2020)</b>',
    labels={'ds': 'Ano/Mês', 'Toneladas': 'Toneladas Coletadas', 'Modelo': 'Série / Modelo'},
    color_discrete_map={
        'Real (2019-2020)': '#111111',  # Linha Preta para o Dado Real
        'Random Forest': '#2ca02c',      # Verde
        'ARIMA': '#ff7f0e',              # Laranja
        'SARIMA': '#1f77b4'              # Azul
    }
)

# Estilização visual
fig_val.update_traces(line=dict(width=2.5))
fig_val.update_layout(
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(showgrid=True, dtick="M1", tickformat="%b/%Y"),
    yaxis=dict(showgrid=True, tickformat=",.0f"),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig_val.show()